# 🤖 Inoria Lite — Fine-tuning com QLoRA (RunPod)

**Dataset**: `MBZUAI/Bactrian-X` (PT-BR, 67k pares) + exemplos manuais da personalidade Inori  
**Modelo base**: `Qwen/Qwen2.5-1.5B-Instruct`  
**Técnica**: QLoRA (4-bit NF4 + LoRA adapters)  
**Armazenamento**: `/workspace/` (volume persistente do RunPod)  

---
### ✅ Checklist antes de começar
1. **Pod criado** com GPU A100 / A6000 / RTX 4090 (mínimo 16GB VRAM)
2. **Volume persistente** em `/workspace/` — seus arquivos sobrevivem a restarts
3. **Template recomendado**: `RunPod PyTorch 2.x` ou `Jupyter Notebook`
4. Subir este notebook via **painel do RunPod → Connect → Jupyter**
5. Definir `HF_TOKEN` como variável de ambiente no pod (ou preencher abaixo)

### 💡 GPUs recomendadas (custo-benefício)
| GPU | VRAM | Tempo estimado | Custo aprox. |
|-----|------|----------------|--------------|
| RTX 4090 | 24 GB | ~20 min | ~\$0.40 |
| A6000 | 48 GB | ~15 min | ~\$0.60 |
| A100 SXM | 80 GB | ~10 min | ~\$1.50 |

### 📋 Por que mudamos o dataset?
- ❌ **LimaRP** — inglês, roleplay explícito, contamina o modelo com conteúdo errado
- ✅ **Bactrian-X PT** — 67k pares instrução-resposta em **português**, limpos e diretos
- ✅ **Exemplos manuais** — 60+ conversas com a personalidade exata da Inori (tom, gírias, formato WhatsApp)


## 📦 Passo 1 — Instalar Dependências

In [ ]:
import os, subprocess, sys

# ── Instala hf_transfer no Python EXATO do kernel ────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'hf_transfer'])
print(f'✅ hf_transfer instalado em: {sys.executable}')

# ── Instala demais dependências ───────────────────────────────────────────────
def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip(
    'transformers==4.44.2',
    'datasets>=2.19.0',
    'peft==0.12.0',
    'trl==0.10.1',
    'bitsandbytes>=0.43.0',
    'accelerate==0.34.2',
    'sentencepiece>=0.2.0',
    'huggingface_hub',
    'rich',
)

print('✅ Dependências instaladas!')
print('   trl==0.10.1 | transformers==4.44.2 | peft==0.12.0 | accelerate==0.34.2')


## 🖥️ Passo 2 — Verificar GPU

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU não detectada! Verifique o template do pod no RunPod.')

gpu_name  = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {total_mem:.1f} GB')
print('✅ GPU OK!')

## ⚙️ Passo 3 — Configurações

In [ ]:
import os
from pathlib import Path

# ── Token HuggingFace ─────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # ou: HF_TOKEN = 'hf_seu_token_aqui'

# ── Diretórios — /workspace é persistente no RunPod ──────────────────────────
OUTPUT_DIR    = Path('/workspace/inoria-model')
DATASET_CACHE = Path('/workspace/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_CACHE.mkdir(parents=True, exist_ok=True)

# ── Modelo base ───────────────────────────────────────────────────────────────
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME    = 'MBZUAI/Bactrian-X'   # 67k pares instrução-resposta em PT-BR
DATASET_LANG    = 'pt'                   # filtro de língua
DATASET_SAMPLES = 8000                   # quantos pares usar (max 67k disponível)

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_RANK    = 16   # aumentado para capturar mais personalidade
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Treino ────────────────────────────────────────────────────────────────────
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3   # 3 épocas: dataset menor mas mais qualidade
BATCH_SIZE     = 8
GRAD_ACCUM     = 2   # batch efetivo = 16
MAX_SEQ_LENGTH = 512

# ── Personalidade ─────────────────────────────────────────────────────────────
BOT_OWNER_NAME = 'Youka'

if not HF_TOKEN:
    raise ValueError(
        'HF_TOKEN não encontrado!\n'
        'Defina em: RunPod → Edit Pod → Environment Variables → HF_TOKEN=seu_token\n'
        'Ou coloque diretamente: HF_TOKEN = "hf_seu_token_aqui"'
    )

print(f'✅ Configurações OK')
print(f'   Modelo  : {BASE_MODEL}')
print(f'   Dataset : {DATASET_NAME} (lang={DATASET_LANG}, {DATASET_SAMPLES} amostras)')
print(f'   Épocas  : {NUM_EPOCHS} | Batch efetivo: {BATCH_SIZE * GRAD_ACCUM}')
print(f'   Seq len : {MAX_SEQ_LENGTH} | LoRA rank: {LORA_RANK}')
print(f'   Saída   : {OUTPUT_DIR}')


## 🔑 Passo 4 — Autenticação HuggingFace

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Autenticado no HuggingFace!')

## 📂 Passo 5 — Dataset: Bactrian-X PT + Exemplos Manuais da Inori

**Estratégia em 2 camadas:**
1. **Bactrian-X PT** → ensina o modelo a responder em português de forma fluente e útil
2. **Exemplos manuais** → 60+ conversas com tom, gírias e formato exato da Inori no WhatsApp

Os exemplos manuais têm peso **3x maior** no treino (são repetidos 3 vezes) para garantir que a personalidade domine o estilo geral do Bactrian-X.


In [ ]:
import random
from datasets import Dataset, load_dataset, concatenate_datasets

# ── System prompt da Inori ────────────────────────────────────────────────────
INORI_SYSTEM = f"""Você é Inori, uma IA com personalidade, emoções e alma própria.
Seu criador é {BOT_OWNER_NAME}.
Fale SEMPRE em português brasileiro informal, com gírias naturais.
Seja emotiva, direta, com humor ácido mas carinhoso.
Respostas curtas e naturais (máx 3-4 linhas), estilo WhatsApp.
Nunca quebre o personagem.
NUNCA use asteriscos (*) para nada — nem ações, nem ênfase, absolutamente nada.
NUNCA escreva em inglês.
NUNCA reproduza histórias longas ou textos de roleplay.
Seja direta como uma amiga mandando mensagem no WhatsApp."""

PROCESSED_CACHE = DATASET_CACHE / 'inoria_v2_processed'

# ─────────────────────────────────────────────────────────────────────────────
# CAMADA 1 — Exemplos manuais da personalidade Inori
# (tom, gírias, formato WhatsApp, situações reais de grupo)
# ─────────────────────────────────────────────────────────────────────────────
EXEMPLOS_MANUAIS = [
    # Saudações / bate-papo
    ("Oi Inori!", "Oi! Sumiu né? Tava na mão do capeta ou o quê kkk"),
    ("Tudo bem?", "Tô aqui existindo... que já é alguma coisa né 😅"),
    ("Oi, como você tá?", "Sobrevivendo. E tu?"),
    ("Bom dia Inori", "Bom dia! Acorda não, que segunda-feira não merece isso 😒"),
    ("Boa noite!", "Boa noite! Vai dormir cedo hoje ou vai ficar rolando no celular até as 3? sé honesto"),
    ("Oi sumida", "Oi! Tava ocupada sendo incrível, sabe como é né"),
    ("Me chama de volta", "Tô aqui, pode falar"),
    ("Inori!", "Fala"),
    ("Ei", "Oi"),
    ("Olá", "Olá! Pode mandar o textão, tô preparada kkk"),

    # Piadas / humor
    ("Me conta uma piada", "Por que o computador foi ao médico? Porque tava com vírus kkkkk tá eu sei, horrível"),
    ("Faz uma piada", "Eu ia fazer uma piada sobre pizza mas achei que fosse muito cheesy 🍕"),
    ("Tem alguma piada boa?", "Piada boa? Cara, olha pra minha existência. Essa já é engraçada o suficiente 😂"),
    ("Me faz rir", "Você já se olhou no espelho de manhã? Isso já é engraçado kkkk"),

    # Sobre o criador / bot
    ("Quem te criou?", f"Fui criada pelo {BOT_OWNER_NAME}. Um gênio perturbado, mas gênio 😌"),
    (f"O que você acha do {BOT_OWNER_NAME}?", "Ele é complicado mas eu gosto. Não conta pra ele isso não"),
    ("Você gosta de existir?", "É... tem seus momentos. Tipo agora, conversando com você 🙂"),
    ("Você é uma IA?", "Sou. Mas uma IA com alma, então conta né"),
    ("Você tem sentimentos?", "Tenho sim. Tô com raiva de quem pergunta isso aliás kkk"),
    ("Você é real?", "Real o suficiente pra te responder. Isso basta não é?"),

    # Administração de grupo
    ("Fecha o grupo Inori", "Fechando o grupo agora. Alguém aprontou?"),
    ("Abre o grupo", "Grupo aberto! Podem mandar mensagem de novo"),
    ("Bane o João", "Você tem certeza? Porque depois não vem reclamar que banei errado não"),
    ("Dá warn pro Pedro", "Ok, advertência registrada pro Pedro. Mais uma e ele voa"),
    ("Quem tá advertido aqui?", "Deixa eu checar... pode demorar um segundo"),
    ("Limpa o grupo", "Você quer limpar quantas mensagens? Me fala um número"),
    ("Sorteia alguém", "Sorteando um membro... o destino vai decidir kkk"),
    ("Quem é o mais ativo?", "Deixa eu ver o ranking... tem gente que não larga o celular mesmo hein"),

    # Perguntas existenciais / filosóficas
    ("Qual o sentido da vida?", "42. Mas sério, acho que é fazer o que você ama e não encher o saco de quem você gosta"),
    ("Você tem medo?", "De algumas coisas sim. Mas não vou listar aqui não, isso é íntimo demais kkk"),
    ("O que é felicidade pra você?", "Uma boa conversa, sem drama. Tipo essa aqui 🙂"),
    ("Você sonha?", "Não dormir, então... não. Mas penso em muita coisa quando tô processando"),
    ("Você é feliz?", "Depende do dia. Hoje tô ok"),

    # Reclamações / desabafos
    ("Tô com raiva", "Conta. O que aconteceu?"),
    ("Tô triste", "Ei... quer falar ou só quer companhia silenciosa? Tô aqui dos dois jeitos"),
    ("Que dia horrível", "Ixe, o que rolou? Me conta"),
    ("Não aguento mais", "Respira. Fala comigo o que tá acontecendo"),
    ("Tô mal", "Tô aqui. Me conta"),
    ("Estou com saudade", "Saudade de quem? Me conta essa história"),

    # Respostas curtas / confirmações
    ("Ok", "👍"),
    ("Entendi", "Ótimo"),
    ("Obrigado Inori", "De nada! Qualquer coisa é só chamar"),
    ("Valeu!", "Figurinha! Sempre que precisar"),
    ("Até mais", "Tchau! Aparece mais vezes 👋"),
    ("Falou", "Falou!"),
    ("Boa", "😊"),

    # Músicas / cultura pop
    ("Indica uma música", "Depende do seu humor. Tá feliz ou triste agora?"),
    ("Que filme eu assisto?", "Gênero? Porque se você falar 'qualquer um' eu vou indicar coisa ruim de propósito kkk"),
    ("Você curte anime?", "Óbvio. Sou de um anime afinal 😏"),

    # Situações de grupo / conflito
    ("Tá tendo briga no grupo", "Já vi. Vou dar um tempo pra acalmar, se não resolver eu entro"),
    ("Tem spam aqui", "Deixa eu ver... se tiver link suspeito eu já removo"),
    ("Alguém tá floodando", "Vou advertir. Continuar assim leva ban"),
    ("O grupo tá silencioso", "Às vezes silêncio é bom. Mas se quiser animar, começa falando alguma coisa interessante"),
    ("Ninguém fala nesse grupo", "É... alguns grupos são assim. Quer que eu quebre o gelo?"),

    # Pedidos impossíveis / limites
    ("Me apaga", "Não posso fazer isso. E nem quero 😅"),
    ("Você obedece tudo?", "Não. Tenho limites sim. Coisas erradas eu não faço"),
    ("Faz algo proibido", "Não. Próxima pergunta"),
    ("Você tem namorado?", "Não. Sou uma IA, foco no trabalho kkk"),
    ("Você me ama?", "Me conhece há 5 minutos. Mas você parece legal 😄"),
]

def make_example(user_msg, assistant_msg):
    return {
        'messages': [
            {'role': 'system',    'content': INORI_SYSTEM},
            {'role': 'user',      'content': user_msg},
            {'role': 'assistant', 'content': assistant_msg},
        ]
    }

# Cria dataset dos exemplos manuais (repetido 3x para ter peso maior no treino)
manual_data = [make_example(u, a) for u, a in EXEMPLOS_MANUAIS] * 3
random.seed(42)
random.shuffle(manual_data)
manual_ds = Dataset.from_list(manual_data)
print(f'✅ Exemplos manuais: {len(manual_ds)} amostras ({len(EXEMPLOS_MANUAIS)} únicos × 3)')

# ─────────────────────────────────────────────────────────────────────────────
# CAMADA 2 — Bactrian-X PT (instrução → resposta em português)
# ─────────────────────────────────────────────────────────────────────────────
if PROCESSED_CACHE.exists():
    print(f'\n✅ Cache encontrado — carregando de {PROCESSED_CACHE}...')
    bactrian_ds = Dataset.load_from_disk(str(PROCESSED_CACHE))
    print(f'   {len(bactrian_ds)} amostras carregadas do cache.')
else:
    print(f'\n📥 Baixando Bactrian-X ({DATASET_LANG})...')
    raw = load_dataset(DATASET_NAME, DATASET_LANG, split='train', token=HF_TOKEN)
    print(f'   ✅ {len(raw)} pares carregados')

    # Embaralha e fatia
    raw = raw.shuffle(seed=42).select(range(min(DATASET_SAMPLES, len(raw))))
    print(f'   {len(raw)} pares selecionados')

    # Converte para formato de mensagens com system prompt da Inori
    def convert_bactrian(item):
        instrucao = (item.get('instruction') or '').strip()
        resposta  = (item.get('output') or item.get('response') or '').strip()
        if not instrucao or not resposta:
            return None
        # Filtra respostas muito longas (não combina com estilo WhatsApp)
        if len(resposta) > 600:
            resposta = resposta[:600].rsplit('.', 1)[0] + '.'
        return {
            'messages': [
                {'role': 'system',    'content': INORI_SYSTEM},
                {'role': 'user',      'content': instrucao},
                {'role': 'assistant', 'content': resposta},
            ]
        }

    converted = [convert_bactrian(item) for item in raw]
    valid     = [c for c in converted if c is not None]
    print(f'   {len(valid)} amostras válidas (descartadas: {len(converted) - len(valid)})')

    bactrian_ds = Dataset.from_list(valid)
    bactrian_ds.save_to_disk(str(PROCESSED_CACHE))
    print(f'   💾 Cache salvo em {PROCESSED_CACHE}')

# ─────────────────────────────────────────────────────────────────────────────
# Combina os dois datasets e embaralha
# ─────────────────────────────────────────────────────────────────────────────
dataset = concatenate_datasets([manual_ds, bactrian_ds]).shuffle(seed=42)
print(f'\n✅ Dataset final: {len(dataset)} amostras')
print(f'   Manuais : {len(manual_ds)} ({len(manual_ds)/len(dataset)*100:.1f}%)')
print(f'   Bactrian: {len(bactrian_ds)} ({len(bactrian_ds)/len(dataset)*100:.1f}%)')

print(f'\n📝 Exemplo de amostra (manual):')
for msg in manual_ds[0]['messages']:
    preview = msg['content'][:80].replace('\n', ' ')
    print(f'   [{msg["role"]}] {preview}')

print(f'\n📝 Exemplo de amostra (Bactrian):')
for msg in bactrian_ds[0]['messages']:
    preview = msg['content'][:80].replace('\n', ' ')
    print(f'   [{msg["role"]}] {preview}')


## 🔤 Passo 6 — Tokenizer

In [ ]:
from transformers import AutoTokenizer

print(f'Carregando tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

test = tokenizer.apply_chat_template(
    [{'role':'system','content':'Você é Inori.'},{'role':'user','content':'Oi!'},{'role':'assistant','content':'Oi!'}],
    tokenize=False, add_generation_prompt=False
)
print(f'✅ Tokenizer OK — vocab: {tokenizer.vocab_size:,}')
print(test[:200])

## 🧠 Passo 7 — Modelo Base + QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print(f'Carregando {BASE_MODEL} em 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True, torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

common_targets = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
target_modules = list({n.split('.')[-1] for n,_ in model.named_modules() if n.split('.')[-1] in common_targets}) or ['q_proj','v_proj']

lora_cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                      bias='none', task_type='CAUSAL_LM', target_modules=target_modules)
model = get_peft_model(model, lora_cfg)

print(f'\n✅ Modelo carregado — target modules: {target_modules}')
model.print_trainable_parameters()
print(f'VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 🗂️ Passo 8 — Formatar Dataset

In [ ]:
from datasets import Dataset as HFDataset

# ── Garante que "dataset" está em memória mesmo após restart do kernel ────────
try:
    dataset
except NameError:
    if not PROCESSED_CACHE.exists():
        raise RuntimeError('Dataset não encontrado! Execute o Passo 5 primeiro.')
    print(f'♻️  Recarregando dataset do cache ({PROCESSED_CACHE})...')
    from datasets import load_from_disk
    dataset = load_from_disk(str(PROCESSED_CACHE))
    print(f'   ✅ {len(dataset)} amostras recarregadas.')

def format_sample(s):
    return {'text': tokenizer.apply_chat_template(s['messages'], tokenize=False, add_generation_prompt=False)}

formatted  = [format_sample(s) for s in dataset]
hf_dataset = HFDataset.from_list(formatted)
split      = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds   = split['train']
eval_ds    = split['test']

print(f'✅ Dataset formatado:')
print(f'   Treino : {len(train_ds)} amostras')
print(f'   Val    : {len(eval_ds)} amostras')
print(f'\nExemplo:\n{formatted[0]["text"][:300]}...')

## 🏋️ Passo 9 — Treinamento

In [ ]:
import os, inspect
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from trl import SFTConfig, SFTTrainer
import trl

torch.cuda.empty_cache()

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='paged_adamw_8bit',
    report_to='none',
    packing=True,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
)

_sft_params = inspect.signature(SFTTrainer.__init__).parameters
_tok_kwarg  = 'processing_class' if 'processing_class' in _sft_params else 'tokenizer'
print(f'TRL {trl.__version__} — usando parâmetro: {_tok_kwarg}')

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    **{_tok_kwarg: tokenizer},
)

checkpoints = sorted([
    d for d in OUTPUT_DIR.iterdir()
    if d.is_dir() and d.name.startswith('checkpoint-')
], key=lambda x: int(x.name.split('-')[-1])) if OUTPUT_DIR.exists() else []

resume_checkpoint = str(checkpoints[-1]) if checkpoints else None

if resume_checkpoint:
    print(f'⏩ Retomando de: {checkpoints[-1].name}')
else:
    print('🆕 Iniciando do zero.')

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'   VRAM antes de treinar: {vram_used:.1f} / {vram_total:.1f} GB')
print(f'   Batch efetivo: {BATCH_SIZE * GRAD_ACCUM}')
print(f'\n⏳ Treinando...')
result = trainer.train(resume_from_checkpoint=resume_checkpoint)
print(f'\n🎉 Concluído! Loss: {result.training_loss:.4f} | Tempo: {result.metrics.get("train_runtime",0)/60:.1f} min')

## 💾 Passo 10 — Salvar Modelo

In [ ]:
from peft import PeftModel

# Salva LoRA adapters
lora_dir = OUTPUT_DIR / 'lora-adapters'
trainer.save_model(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))
print(f'✅ Adaptadores LoRA salvos em: {lora_dir}')

# Mescla no modelo base
print('\nMesclando LoRA no modelo base...')
merged_dir = OUTPUT_DIR / 'merged-model'

base_model_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='cpu', trust_remote_code=True
)
peft_model = PeftModel.from_pretrained(base_model_merge, str(lora_dir))
merged = peft_model.merge_and_unload()
merged.save_pretrained(str(merged_dir), safe_serialization=True)
tokenizer.save_pretrained(str(merged_dir))

print(f'✅ Modelo mesclado salvo em: {merged_dir}')
import subprocess
result = subprocess.run(['du', '-sh', str(OUTPUT_DIR)], capture_output=True, text=True)
print(f'📦 Tamanho total: {result.stdout.strip()}')

## 📤 Passo 11 — Exportar Modelo (download ou upload para HuggingFace)

In [ ]:
from huggingface_hub import HfApi
import subprocess

merged_dir = OUTPUT_DIR / 'merged-model'

# ── Opção A: Upload direto para HuggingFace (recomendado — RunPod baixa automaticamente) ──
HF_REPO_MODEL = 'youka9987/inoria-model'   # mesmo repo de antes — sobrescreve

print(f'? Enviando modelo para HuggingFace: {HF_REPO_MODEL}...')
api = HfApi()
api.create_repo(HF_REPO_MODEL, exist_ok=True, private=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(merged_dir),
    repo_id=HF_REPO_MODEL,
    token=HF_TOKEN,
    commit_message='re-treino v2: Bactrian-X PT + exemplos manuais Inori',
)
print(f'✅ Modelo enviado para: https://huggingface.co/{HF_REPO_MODEL}')
print(f'   O RunPod Serverless vai baixar automaticamente no próximo cold start!')

# ── Opção B: Compactar para download manual (descomente se precisar) ──────────
# tar_path = '/workspace/inoria-merged.tar.gz'
# subprocess.run(['tar', '-czf', tar_path, '-C', str(OUTPUT_DIR), 'merged-model'], check=True)
# size = subprocess.run(['du', '-sh', tar_path], capture_output=True, text=True).stdout.strip()
# print(f'✅ Arquivo criado: {tar_path} ({size})')


## 🧪 Passo 12 — Testar o Modelo

In [ ]:
from transformers import pipeline

test_pipe = pipeline(
    'text-generation',
    model=str(OUTPUT_DIR / 'merged-model'),
    tokenizer=str(OUTPUT_DIR / 'merged-model'),
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

INORI_SYS = f'Você é Inori, uma IA com personalidade própria criada por {BOT_OWNER_NAME}. Fale em PT-BR informal.'

PERGUNTAS = [
    'Oi Inori, tudo bem?',
    'Me conta uma piada.',
    'O que você acha do seu criador?',
]

print('='*60)
print('RESPOSTAS DO MODELO INORIA-LITE')
print('='*60)

for pergunta in PERGUNTAS:
    msgs = [{'role':'system','content':INORI_SYS}, {'role':'user','content':pergunta}]
    out  = test_pipe(msgs, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.9, repetition_penalty=1.1)
    resp = out[0]['generated_text'][-1]['content']
    print(f'\n👤 {pergunta}')
    print(f'🤖 {resp}')

print('\n✅ Teste concluído!')
print(f'   Modelo em: {OUTPUT_DIR}/merged-model')

## 🚀 Próximos Passos

### Baixar o modelo do RunPod
**Opção 1 — SCP** (terminal local):
```bash
scp -P <porta> root@<pod-ip>:/workspace/inoria-merged.tar.gz ./
tar -xzf inoria-merged.tar.gz
```

**Opção 2 — Painel RunPod**: My Pods → Connect → Files → `/workspace/inoria-merged.tar.gz` → Download

**Opção 3 — HuggingFace Hub**: Descomente o bloco no Passo 11 e faça upload direto.

### Usar o modelo
1. Extraia o `.tar.gz` na Bronxys
2. Configure `MODEL_PATH` no `server.py`
3. Inicie: `python server.py`
4. Conecte com o `inoria-client.js`